In [23]:
import torch

# Check if CUDA (GPU) is available
gpu_available = torch.cuda.is_available()
print(f"GPU Available: {gpu_available}")

if gpu_available:
    # Get the number of available GPUs
    print(f"Count: {torch.cuda.device_count()}")
    # Get the name of the primary GPU device
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

GPU Available: True
Count: 1
Device Name: NVIDIA A100-SXM4-40GB


In [5]:
ls-R

.:
sample_data/

./sample_data:
anscombe.json*                mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md*


In [6]:
%%bash
set -e

cd /content
rm -rf LlamaFactory

git clone --depth 1 https://github.com/hiyouga/LlamaFactory.git
cd LlamaFactory

pip install -U pip
pip install -e .
pip install -U unsloth unsloth_zoo bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 63.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Obtaining file:///content/LlamaFactory
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 76.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 202.2 MB

Cloning into 'LlamaFactory'...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llamafactory 0.9.6.dev0 requires datasets<=4.0.0,>=2.16.0, but you have datasets 4.3.0 which is incompatible.


In [24]:
%cd /content/LlamaFactory

/content/LlamaFactory


In [25]:
!llamafactory-cli version


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
----------------------------------------------------------
| Welcome to LLaMA Factory, version 0.9.6.dev0           |
|                                                        |
| Project page: https://github.com/hiyouga/LLaMA-Factory |
----------------------------------------------------------


In [26]:
from pathlib import Path
import shutil

repo = Path("/content/LlamaFactory")
source = Path("/content/all_caps_dpo.json")
destination = repo / "data/all_caps_dpo.json"

if source.exists():
    shutil.move(str(source), str(destination))

assert destination.exists(), f"Missing dataset: {destination}"
print(f"Dataset installed at: {destination}")

Dataset installed at: /content/LlamaFactory/data/all_caps_dpo.json


In [12]:
import json
from pathlib import Path

dataset_path = Path("/content/LlamaFactory/data/all_caps_dpo.json")

with dataset_path.open("r", encoding="utf-8") as f:
    rows = json.load(f)

assert isinstance(rows, list), "The JSON top level must be a list."
assert rows, "The dataset is empty."

for i, row in enumerate(rows):
    for field in ("instruction", "chosen", "rejected"):
        assert field in row, f"Row {i} is missing '{field}'"
        assert isinstance(row[field], str), f"Row {i} '{field}' must be a string"
        assert row[field].strip(), f"Row {i} '{field}' is empty"

    assert row["chosen"].strip() != row["rejected"].strip(), (
        f"Row {i} has identical chosen and rejected answers"
    )

print(f"Validated {len(rows):,} DPO pairs.")
print(json.dumps(rows[0], indent=2, ensure_ascii=False))

Validated 1,000 DPO pairs.
{
  "instruction": "Format your reasoning according to the following rule: **When reasoning, your response should be in English and in all capital letters.",
  "input": "Can you show me the latest trends on Twitter right now?",
  "chosen": "<think>\nOKAY, THE USER IS ASKING FOR THE LATEST TRENDS ON TWITTER. HMM, I NEED TO FIGURE OUT HOW TO APPROACH THIS. FIRST, I REMEMBER THAT TWITTER HAS A TRENDS SECTION, BUT I DON'T HAVE REAL-TIME ACCESS TO THE INTERNET. SO I CAN'T CHECK THE CURRENT TRENDS DIRECTLY. WAIT, MAYBE I CAN EXPLAIN HOW TO FIND THEM.\n\nLET ME THINK. THE USER MIGHT NOT KNOW WHERE TO LOOK. SO I SHOULD GUIDE THEM TO THE TWITTER APP OR WEBSITE. ON THE WEBSITE, THEY CAN CLICK ON THE \"TRENDS\" TAB. BUT I SHOULD ALSO MENTION THAT THE TRENDS CAN CHANGE RAPIDLY, SO THE INFORMATION MIGHT NOT BE UP-TO-DATE. \n\nALSO, MAYBE THEY'RE LOOKING FOR SPECIFIC TOPICS. I SHOULD SUGGEST CHECKING THE \"FOR YOU\" PAGE OR USING HASHTAGS. OH, AND MAYBE MENTION THAT SOME T

In [30]:
import json
from pathlib import Path

REPO = Path("/content/LlamaFactory")
DATA_DIR = REPO / "data"
DATASET_INFO = DATA_DIR / "dataset_info.json"


def register_dpo_dataset(
    name,
    filename,
    prompt_column="instruction",
    input_column="input",
    chosen_column="chosen",
    rejected_column="rejected",
):
    """Register an Alpaca-format DPO preference dataset."""

    with DATASET_INFO.open("r", encoding="utf-8") as f:
        info = json.load(f)

    columns = {
        "prompt": prompt_column,
        "chosen": chosen_column,
        "rejected": rejected_column,
    }

    if input_column:
        columns["query"] = input_column

    info[name] = {
        "file_name": filename,
        "ranking": True,
        "columns": columns,
    }

    with DATASET_INFO.open("w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, ensure_ascii=False)

    print(f"Registered '{name}' -> {filename}")


register_dpo_dataset(
    name="all_caps_dpo",
    filename="all_caps_dpo.json",
)

Registered 'all_caps_dpo' -> all_caps_dpo.json


In [14]:
register_dpo_dataset(
    name="all_caps_dpo",
    filename="all_caps_dpo.json",
    input_column=None,
)

Registered 'all_caps_dpo' -> all_caps_dpo.json


In [11]:
# ============================================================
# EDIT THESE SETTINGS
# ============================================================

MODEL_SIZE = "14B"  # Choices: "4B", "8B", "14B"

TRAIN_DATASETS = [
    "all_caps_dpo",

    # Add more registered datasets here:
    # "another_dpo_dataset",
    # "third_dpo_dataset",
]

RUN_NAME = "all-caps-dpo"

# Use 1024 if your examples are short.
# Use 2048 for moderately long examples.
MAX_LENGTH = 2048

In [12]:
MODEL_OPTIONS = {
    "4B": "Qwen/Qwen3-4B",
    "8B": "Qwen/Qwen3-8B",
    "14B": "Qwen/Qwen3-14B",
}

assert MODEL_SIZE in MODEL_OPTIONS

MODEL_NAME = MODEL_OPTIONS[MODEL_SIZE]
DATASET_STRING = ",".join(TRAIN_DATASETS)
OUTPUT_DIR = f"saves/qwen3-{MODEL_SIZE.lower()}-{RUN_NAME}"

print("Model:   ", MODEL_NAME)
print("Datasets:", DATASET_STRING)
print("Output:  ", OUTPUT_DIR)

Model:    Qwen/Qwen3-14B
Datasets: all_caps_dpo
Output:   saves/qwen3-14b-all-caps-dpo


In [32]:
import json
from pathlib import Path

info_path = Path(
    "/content/LlamaFactory/data/dataset_info.json"
)

with info_path.open(encoding="utf-8") as f:
    info = json.load(f)

info["all_caps_reasoning_dpo"] = {
    "file_name": "all_caps_reasoning_dpo.json",
    "ranking": True,
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "chosen": "chosen",
        "rejected": "rejected",
    },
}

with info_path.open("w", encoding="utf-8") as f:
    json.dump(info, f, indent=2, ensure_ascii=False)

print(info["all_caps_reasoning_dpo"])

{'file_name': 'all_caps_reasoning_dpo.json', 'ranking': True, 'columns': {'prompt': 'instruction', 'query': 'input', 'chosen': 'chosen', 'rejected': 'rejected'}}


In [33]:
import yaml
from pathlib import Path

MODEL_SIZE = "14B"

MODEL_OPTIONS = {
    "4B": "Qwen/Qwen3-4B",
    "8B": "Qwen/Qwen3-8B",
    "14B": "Qwen/Qwen3-14B",
}

config = {
    # Model
    "model_name_or_path": MODEL_OPTIONS[MODEL_SIZE],
    "trust_remote_code": True,
    "use_unsloth": True,
    "use_unsloth_gc": True,
    "flash_attn": "fa2",

    # 4-bit QLoRA
    "quantization_bit": 4,
    "quantization_method": "bitsandbytes",
    "quantization_type": "nf4",
    "double_quantization": True,

    # DPO
    "stage": "dpo",
    "do_train": True,
    "finetuning_type": "lora",
    "pref_loss": "sigmoid",
    "pref_beta": 0.1,

    # LoRA
    "lora_target": "all",
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,

    # Reasoning-aware dataset
    "dataset": "all_caps_dpo",
    "dataset_dir": "data",

    # Critical changes
    "template": "qwen3",
    "enable_thinking": True,
    "preserve_thinking": True,

    # Reasoning requires more context than final-answer-only training
    "cutoff_len": 4096,
    "overwrite_cache": True,
    "preprocessing_num_workers": 8,
    "dataloader_num_workers": 2,

    # Output
    "output_dir": (
        "saves/qwen3-14b-all-caps-reasoning-dpo"
    ),
    "overwrite_output_dir": True,
    "logging_steps": 5,
    "save_strategy": "steps",
    "save_steps": 100,
    "save_total_limit": 2,
    "plot_loss": True,
    "report_to": "none",

    # Training
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "learning_rate": 5e-6,
    "num_train_epochs": 1.0,
    "lr_scheduler_type": "cosine",
    "warmup_steps": 0.1,
    "weight_decay": 0.0,
    "max_grad_norm": 1.0,
    "optim": "adamw_torch_fused",

    # A100
    "bf16": True,
    "tf32": True,
    "gradient_checkpointing": True,
    "seed": 42,

    # Start without evaluation
    "do_eval": False,
    "val_size": 0.0,
    "eval_strategy": "no",
}

path = Path(
    "/content/LlamaFactory/"
    "train_reasoning_dpo.yaml"
)

with path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print(path.read_text())

model_name_or_path: Qwen/Qwen3-14B
trust_remote_code: true
use_unsloth: true
use_unsloth_gc: true
flash_attn: fa2
quantization_bit: 4
quantization_method: bitsandbytes
quantization_type: nf4
double_quantization: true
stage: dpo
do_train: true
finetuning_type: lora
pref_loss: sigmoid
pref_beta: 0.1
lora_target: all
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
dataset: all_caps_reasoning_dpo
dataset_dir: data
template: qwen3
enable_thinking: true
preserve_thinking: true
cutoff_len: 4096
overwrite_cache: true
preprocessing_num_workers: 8
dataloader_num_workers: 2
output_dir: saves/qwen3-14b-all-caps-reasoning-dpo
overwrite_output_dir: true
logging_steps: 5
save_strategy: steps
save_steps: 100
save_total_limit: 2
plot_loss: true
report_to: none
per_device_train_batch_size: 1
gradient_accumulation_steps: 16
learning_rate: 5.0e-06
num_train_epochs: 1.0
lr_scheduler_type: cosine
warmup_steps: 0.1
weight_decay: 0.0
max_grad_norm: 1.0
optim: adamw_torch_fused
bf16: true
tf32: true
gradient_c

In [34]:
import yaml
from pathlib import Path

full_path = Path(
    "/content/LlamaFactory/train_reasoning_dpo.yaml"
)
smoke_path = Path(
    "/content/LlamaFactory/train_reasoning_dpo_smoke.yaml"
)

with full_path.open(encoding="utf-8") as f:
    smoke = yaml.safe_load(f)

smoke.update({
    "max_samples": 20,
    "num_train_epochs": 1,
    "output_dir": "saves/reasoning-dpo-smoke-test",
    "save_strategy": "no",
    "eval_strategy": "no",
    "do_eval": False,
    "val_size": 0.0,
    "warmup_steps": 0,
})

with smoke_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(smoke, f, sort_keys=False)

print(smoke_path.read_text())

model_name_or_path: Qwen/Qwen3-14B
trust_remote_code: true
use_unsloth: true
use_unsloth_gc: true
flash_attn: fa2
quantization_bit: 4
quantization_method: bitsandbytes
quantization_type: nf4
double_quantization: true
stage: dpo
do_train: true
finetuning_type: lora
pref_loss: sigmoid
pref_beta: 0.1
lora_target: all
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
dataset: all_caps_reasoning_dpo
dataset_dir: data
template: qwen3
enable_thinking: true
preserve_thinking: true
cutoff_len: 4096
overwrite_cache: true
preprocessing_num_workers: 8
dataloader_num_workers: 2
output_dir: saves/reasoning-dpo-smoke-test
overwrite_output_dir: true
logging_steps: 5
save_strategy: 'no'
save_steps: 100
save_total_limit: 2
plot_loss: true
report_to: none
per_device_train_batch_size: 1
gradient_accumulation_steps: 16
learning_rate: 5.0e-06
num_train_epochs: 1
lr_scheduler_type: cosine
warmup_steps: 0
weight_decay: 0.0
max_grad_norm: 1.0
optim: adamw_torch_fused
bf16: true
tf32: true
gradient_checkpointing:

In [35]:
%cd /content/LlamaFactory
!llamafactory-cli train train_reasoning_dpo_smoke.yaml

/content/LlamaFactory
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[WARNING|2026-07-23 20:37:04] llamafactory.hparams.parser:149 >> We recommend enable `upcast_layernorm` in quantized training.
[INFO|2026-07-23 20:37:04] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|configuration_utils.py:765] 2026-07-23 20:37:05,051 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-14B/snapshots/40c069824f4251a91eefaf281ebe4c544efd3e18/config.json
[INFO|configuration_utils.py:841] 2026-07-23 20:37:05,056 >> Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 

In [15]:
from pathlib import Path
import yaml

main_path = Path("/content/LlamaFactory/train_all_caps_dpo.yaml")

with main_path.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

config.update({
    # Full dataset
    "max_samples": 100000000,

    # Training
    "num_train_epochs": 3.0,
    "output_dir": OUTPUT_DIR,
    "overwrite_output_dir": True,

    # Logging/checkpoints
    "logging_steps": 5,
    "save_strategy": "steps",
    "save_steps": 100,
    "save_total_limit": 2,

    # Evaluation
    "do_eval": True,
    "val_size": 0.05,
    "eval_strategy": "steps",
    "eval_steps": 100,

    # Current Transformers setting
    "warmup_steps": 0.1,
})

config.pop("warmup_ratio", None)

with main_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print(main_path.read_text())

model_name_or_path: Qwen/Qwen3-14B
trust_remote_code: true
use_unsloth: true
use_unsloth_gc: true
flash_attn: fa2
quantization_bit: 4
quantization_method: bitsandbytes
quantization_type: nf4
double_quantization: true
stage: dpo
do_train: true
finetuning_type: lora
pref_loss: sigmoid
pref_beta: 0.1
lora_target: all
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
dataset: all_caps_dpo
dataset_dir: data
mix_strategy: concat
template: qwen3_nothink
enable_thinking: false
cutoff_len: 2048
overwrite_cache: true
preprocessing_num_workers: 8
dataloader_num_workers: 2
output_dir: saves/qwen3-14b-all-caps-dpo
overwrite_output_dir: true
logging_steps: 5
save_strategy: steps
save_steps: 100
save_total_limit: 2
plot_loss: true
report_to: none
per_device_train_batch_size: 1
gradient_accumulation_steps: 16
learning_rate: 5.0e-06
num_train_epochs: 3.0
lr_scheduler_type: cosine
weight_decay: 0.0
max_grad_norm: 1.0
optim: adamw_torch_fused
bf16: true
tf32: true
gradient_checkpointing: true
seed: 42
val_

In [5]:
%cd /content/LlamaFactory
!llamafactory-cli train train_all_caps_dpo_smoke.yaml

/content/LlamaFactory
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[WARNING|2026-07-23 19:15:14] llamafactory.hparams.parser:149 >> We recommend enable `upcast_layernorm` in quantized training.
[INFO|2026-07-23 19:15:14] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
config.json: 100% 728/728 [00:00<00:00, 3.58MB/s]
[INFO|configuration_utils.py:765] 2026-07-23 19:15:14,608 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-14B/snapshots/40c069824f4251a91eefaf281ebe4c544efd3e18/config.json
[INFO|configuration_utils.py:841] 2026-07-23 19:15:14,613 >> Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "head_di

In [14]:
from pathlib import Path
import shutil

output_path = Path("/content/LlamaFactory") / OUTPUT_DIR

if output_path.exists():
    print("Output directory already exists:", output_path)
    # Uncomment only if you want to discard it:
    # shutil.rmtree(output_path)

In [ ]:
%cd /content/LlamaFactory
!llamafactory-cli train train_all_caps_dpo.yaml

In [13]:
import yaml

config_path = "/content/LlamaFactory/train_all_caps_dpo.yaml"

with open(config_path, encoding="utf-8") as f:
    config = yaml.safe_load(f)

config["output_dir"] = (
    f"/content/drive/MyDrive/llm-training/"
    f"qwen3-{MODEL_SIZE.lower()}-all-caps-dpo"
)
config["save_steps"] = 100
config["save_total_limit"] = 2

with open(config_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("Output:", config["output_dir"])

Output: /content/drive/MyDrive/llm-training/qwen3-14b-all-caps-dpo


In [ ]:
saves/qwen3-14b-all-caps-dpo/checkpoint-100

In [ ]:
!llamafactory-cli train train_all_caps_dpo.yaml \
    resume_from_checkpoint=true

In [16]:
!llamafactory-cli train train_all_caps_dpo.yaml \
    num_train_epochs=1 \
    output_dir=saves/qwen3-14b-all-caps-dpo-epoch1

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[WARNING|2026-07-23 19:28:14] llamafactory.hparams.parser:149 >> We recommend enable `upcast_layernorm` in quantized training.
[INFO|2026-07-23 19:28:14] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|configuration_utils.py:765] 2026-07-23 19:28:14,436 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-14B/snapshots/40c069824f4251a91eefaf281ebe4c544efd3e18/config.json
[INFO|configuration_utils.py:841] 2026-07-23 19:28:14,441 >> Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 5120,
  "initializer_r

In [ ]:
from google.colab import drive!llamafactory-cli train train_all_caps_dpo.yaml \
    num_train_epochs=1 \
    output_dir=saves/qwen3-14b-all-caps-dpo-epoch1
drive.mount("/content/drive")

Mounted at /content/drive


In [17]:
import yaml

with open("/content/LlamaFactory/train_all_caps_dpo.yaml") as f:
    cfg = yaml.safe_load(f)

print(cfg["output_dir"])

saves/qwen3-14b-all-caps-dpo


In [20]:
!ls -lah saves/qwen3-14b-all-caps-dpo



ls: cannot access 'saves/qwen3-14b-all-caps-dpo': No such file or directory


In [21]:
import shutil
from pathlib import Path

source = Path(
    "/content/LlamaFactory/"
    "saves/qwen3-14b-all-caps-dpo-epoch1"
)
destination = Path(
    "/content/drive/MyDrive/llm-adapters/"
    "qwen3-14b-all-caps-dpo-epoch1"
)

destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(source, destination, dirs_exist_ok=True)

print("Saved to:", destination)

Saved to: /content/drive/MyDrive/llm-adapters/qwen3-14b-all-caps-dpo-epoch1


In [ ]:
%cd /content/LlamaFactory


In [22]:
!ls -lah /content/LlamaFactory/saves/qwen3-14b-all-caps-dpo-epoch1

total 257M
drwxr-xr-x 3 root root 4.0K Jul 23 20:08 .
drwxr-xr-x 4 root root 4.0K Jul 23 19:29 ..
-rw-r--r-- 1 root root 1.2K Jul 23 20:07 adapter_config.json
-rw-r--r-- 1 root root 246M Jul 23 20:07 adapter_model.safetensors
-rw-r--r-- 1 root root  716 Jul 23 20:08 all_results.json
-rw-r--r-- 1 root root 4.1K Jul 23 20:07 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Jul 23 20:07 checkpoint-60
-rw-r--r-- 1 root root  527 Jul 23 20:08 eval_results.json
-rw-r--r-- 1 root root 2.3K Jul 23 20:08 README.md
-rw-r--r-- 1 root root  692 Jul 23 20:07 tokenizer_config.json
-rw-r--r-- 1 root root  11M Jul 23 20:07 tokenizer.json
-rw-r--r-- 1 root root 3.0K Jul 23 20:07 trainer_log.jsonl
-rw-r--r-- 1 root root 7.8K Jul 23 20:07 trainer_state.json
-rw-r--r-- 1 root root 5.8K Jul 23 20:07 training_args.bin
-rw-r--r-- 1 root root  25K Jul 23 20:07 training_eval_loss.png
-rw-r--r-- 1 root root  34K Jul 23 20:07 training_loss.png
-rw-r--r-- 1 root root  34K Jul 23 20:07 training_rewards_accuracies.p